# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing a dataset defined by a Croissant schema using the `mlcroissant` library. All references to entities such as record sets, fields, and columns are made using their `@id`.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset name:', metadata.name)
print('Description:', metadata.description)


## 2. Data Overview
List available record sets and their fields with their `@id` identifiers.

In [ ]:
# List all record sets and fields by @id
record_sets = list(dataset.record_sets.keys())
print('Available Record Sets (@id):')
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f'  Record Set @id: {rs_id}')
    print(f'    Fields:')
    for field_id in rs.fields:
        field_obj = dataset.fields[field_id]
        print(f'      Field @id: {field_id} | Name: {field_obj.name} | DataType: {getattr(field_obj, "data_type", "N/A")}')
    print()
# Print a preview of records from the first record set
if record_sets:
    preview_rs_id = record_sets[0]
    print(f'Example records from record set {preview_rs_id}:')
    for i, rec in enumerate(dataset.records(record_set=preview_rs_id)):
        print(rec)
        if i > 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references use `@id`.

In [ ]:
# List of record set @id's
record_sets_ids = record_sets

dataframes = {}
for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f'DataFrame for record set @id: {rs_id}')
        print(f'Columns (@id): {dataframes[rs_id].columns.tolist()}')
        print(dataframes[rs_id].head(), '\n')
    else:
        print(f'The record set {rs_id} is empty or contains no records.')

# For demonstration, select the first available record set for further EDA
main_rs_id = record_sets_ids[0] if record_sets_ids else None
df = dataframes.get(main_rs_id)

## 4. Exploratory Data Analysis (EDA)
Explore, filter, and transform numeric and categorical fields.

Below, always reference fields by their `@id`.

In [ ]:
# Identify field @ids with numeric data for EDA
if df is not None:
    numeric_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    groupable_cols = [col for col in df.columns if df[col].dtype == 'object']
    print('Numeric field @ids:', numeric_cols)
    print('Groupable field @ids:', groupable_cols)
    
    # Example: Use first numeric field and first group field
    numeric_field_id = numeric_cols[0] if numeric_cols else None
    group_field_id = groupable_cols[0] if groupable_cols else None
    print(f'Using numeric field @id: {numeric_field_id}')
    print(f'Using group field @id: {group_field_id}')

    # Example threshold for filtering
    threshold = 10
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f'Filtered records with {numeric_field_id} > {threshold}:')
        print(filtered_df.head())
        
        # Normalize numeric field
        filtered_df[f'{numeric_field_id}_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f'Normalized {numeric_field_id} for filtered records:')
        print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

        # Group by group_field_id if present
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f'Grouped mean {numeric_field_id} by {group_field_id}:')
            print(grouped_df.head())

## 5. Visualization
Visualize numeric data distributions or relationships (with field `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=10, color="dodgerblue")
    plt.title(f'Distribution of numeric field (@id): {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
- This notebook guided users through loading and exploring a Croissant-defined dataset using field and record set `@id` identifiers for reproducible workflows.
- The data inspection and filtering steps can be adapted for more detailed clinical variable analysis, biomarker stratification, or further model-ready transformations.
- Please refer to dataset documentation for specific field mappings, medical terminologies, and recommendations for model training or evaluation.